# MLOps Framework — Detailed Plan & Architecture

> **Status:** DRAFT — Review before build  
> **Author:** Genie Code + sunny.singh@databricks.com  
> **Date:** 2026-06-15  
> **Pattern:** Cookie-cutter (same pattern as Genie Multi-Agent Framework)

---

## 1. Framework Overview

**Purpose:** A reusable, config-driven MLOps framework for tabular ML (classification/regression) that can be deployed across multiple Databricks workspaces with minimal customization.

**Two operational modes:**

| Mode | Use Case | Entry Point |
| --- | --- | --- |
| **Train** | Build a new model from scratch (EDA → tune → deploy) | Full pipeline |
| **Migrate** | Bring an existing model pickle from external system into Databricks MLOps | Wrap → deploy |

**Core principles:**
- Config-driven: swap `config.yaml` for a new use case
- DAB-native: deployable via `databricks bundle deploy` across environments
- INSTALL notebook: human-friendly entry point for understanding + manual runs
- Feature Engineering Client: lineage throughout (features → model → scoring)
- Preprocessing always explicit: never hidden inside a pickle
- Pandas/sklearn for modeling, Spark for data at scale
- Parallel Optuna HPO for faster tuning
- MLflow experiment tracking with full params, metrics, artifacts
- Idempotent: re-runnable without side effects

## 2. Configuration Schema (config.yaml)

The ONLY file users edit. Everything else is generated.

```yaml
# ════════════════════════════════════════
# FRAMEWORK MODE
# ════════════════════════════════════════
mode: "train"                         # "train" | "migrate"

# ════════════════════════════════════════
# COMMON CONFIGURATION (both modes)
# ════════════════════════════════════════
catalog: "my_catalog"
schema: "my_schema"
model_name: "churn_model"
task_type: "classification"            # "classification" | "regression"
entity_key: "customer_id"
timestamp_key: "event_timestamp"       # for point-in-time correctness
label_column: "churned"
feature_table_name: "my_catalog.my_schema.churn_features"

# ════════════════════════════════════════
# TRAIN MODE CONFIGURATION
# ════════════════════════════════════════
train:
  training_table: "my_catalog.my_schema.labeled_customers"
  model_algorithm: "lightgbm"          # "lightgbm" | "xgboost" | "random_forest" | "deep_learning" | "all"
  n_trials: 50                         # Optuna trials
  parallel_trials: "auto"              # "auto" | 1 | 2 | 4 | 8
  split_ratios:
    train: 0.70
    val: 0.15
    test: 0.15
  positive_label: 1                    # classification only

  # Deep learning settings (only used when model_algorithm: "deep_learning" or "all")
  deep_learning:
    framework: "pytorch"               # "pytorch" | "tensorflow"
    architecture: "tabular_mlp"        # "tabular_mlp" | "tabular_resnet" | "custom"
    hidden_layers: [256, 128, 64]      # layer sizes (ignored if architecture: "custom")
    dropout: 0.3                       # dropout rate between layers
    activation: "relu"                 # "relu" | "gelu" | "silu"
    epochs: 100                        # max epochs (early stopping may halt sooner)
    batch_size: 256                    # training batch size
    early_stopping_patience: 10        # epochs without improvement before stopping
    learning_rate_range: [0.0001, 0.01]  # Optuna search range
    weight_decay_range: [0.00001, 0.01]  # L2 regularization range
    compute: "gpu"                     # "cpu" | "gpu" (switches to GPU serverless)

# ════════════════════════════════════════
# MIGRATE MODE CONFIGURATION
# ════════════════════════════════════════
migrate:
  model_path: "/Volumes/my_catalog/my_schema/models/churn_lgbm.pkl"
  model_type: "sklearn"                # "sklearn" | "xgboost" | "lightgbm" | "pytorch"
  reference_table: "my_catalog.my_schema.migration_reference"  # known-good predictions for validation
  additional_artifacts:                # scaler/encoder pickles from source system
    - "/Volumes/my_catalog/my_schema/models/scaler.pkl"
    - "/Volumes/my_catalog/my_schema/models/encoder.pkl"

# ════════════════════════════════════════
# INFERENCE CONFIGURATION
# ════════════════════════════════════════
inference_mode: "both"                 # "serving" | "batch" | "both"

batch:
  output_table: "my_catalog.my_schema.churn_predictions"
  scoring_table: "my_catalog.my_schema.customers_to_score"  # entities to score
  schedule: "0 0 6 * * ? *"           # daily 6am (Quartz cron), or null for manual

serving:
  endpoint_name: "churn_model_serving"  # auto-derived if null
  scale_to_zero: true
  workload_size: "Small"               # "Small" | "Medium" | "Large"
  workload_type: "CPU"                 # "CPU" | "GPU"

# ════════════════════════════════════════
# EXPLAINABILITY CONFIGURATION
# ════════════════════════════════════════
explainability:
  enabled: true                        # Generate explainability notebook
  methods:
    - "shap"                           # SHAP values (global + local)
    - "permutation_importance"         # Permutation feature importance
    - "partial_dependence"             # Partial dependence plots (top-5 features)
  n_samples: 1000                      # Subsample size for SHAP (controls runtime)
  log_to_mlflow: true                  # Log plots as MLflow artifacts

# ════════════════════════════════════════
# MONITORING CONFIGURATION
# ════════════════════════════════════════
monitoring:
  enabled: true
  granularity: "1 day"                 # "1 hour" | "1 day" | "1 week"
  refresh_schedule: "0 0 8 * * ? *"   # daily 8am, or null for manual

# ════════════════════════════════════════
# DEPLOYMENT TARGET
# ════════════════════════════════════════
target: "dev"                          # "dev" | "staging" | "prod"
```

## 3. Generated Directory Structure

```
mlops-framework/
├── INSTALL.py                              ← Entry point notebook (3 cells)
├── config.yaml                             ← THE ONLY FILE USERS EDIT (besides 02_feature_engineering)
├── databricks.yml                          ← Generated DAB config (multi-env)
├── resources/
│   ├── training_pipeline_job.yml            ← One-shot training/migration pipeline
│   └── batch_inference_job.yml              ← Recurring batch scoring job
├── src/
│   └── notebooks/
│       ├── [TRAIN MODE]
│       │   ├── 01_eda.py                       ← Exploratory Data Analysis
│       │   ├── 02_feature_engineering.py        ← User-editable feature logic (with samples)
│       │   ├── 03_train_tune.py                ← Baselines + Optuna HPO (parallel)
│       │   ├── 03b_train_deep_learning.py      ← PyTorch/TF neural network (if DL enabled)
│       │   ├── 04_evaluate.py                  ← Holdout test scoring + audit
│       │   ├── 04b_explainability.py           ← SHAP + feature importance + PDP plots
│       │   └── 05_register.py                  ← UC registration + Champion alias
│       │
│       ├── [MIGRATE MODE]
│       │   ├── 01_validate_model.py            ← Load pickle, smoke test, inspect
│       │   ├── 02_feature_engineering.py        ← Same template as train mode
│       │   └── 03_wrap_and_register.py          ← Pyfunc wrapper + fe.log_model()
│       │
│       ├── [COMMON — both modes]
│       │   ├── 06_batch_inference.py           ← fe.score_batch() + write predictions
│       │   ├── 07_serve.py                     ← Create serving endpoint + inference tables
│       │   ├── 08_monitor.py                   ← Lakehouse Monitoring setup
│       │   └── 09_process_inference.py          ← Flatten serving payload for monitoring
│       │
│       └── [UTILITY]
│           ├── helpers.py                      ← Shared config loader + logging utilities
│           └── nn_architectures.py             ← Predefined DL architectures (MLP, ResNet, custom template)
│
└── deployment/
    └── DEPLOYMENT_LOG.md                   ← Audit trail (auto-appended by framework)
```

**Note:** The INSTALL notebook generates ONLY the relevant notebooks based on `mode`. Train mode skips migrate notebooks and vice versa. Common notebooks (06-09) are always generated based on `inference_mode` and `monitoring.enabled`.

## 4. Notebook Specifications — TRAIN Mode

### 4.1 — 01_eda.py (Exploratory Data Analysis)

| Cell # | Title | What it does | Key outputs |
| --- | --- | --- | --- |
| 1 | Setup & Config | Load config from job params (with widget fallback for interactive use). Install libs if needed. | Config dict available |
| 2 | Load Data | `spark.table(training_table)` → profile shape, dtypes, nulls | Print: "N rows, M columns" |
| 3 | Statistical Profile | `.describe()`, null counts, cardinality per column. Two-column table: feature name → feature type (numeric/categorical/ordinal) | Displayed DataFrame |
| 4 | Target Distribution | Class balance bar chart (classification) or histogram (regression). Flag imbalance if ratio > 5:1 | Chart + imbalance warning |
| 5 | Correlation Analysis | Correlogram (feature-feature). Top-10 correlations with target. At least 2 feature-vs-target visualizations | Heatmap + scatter/box plots |
| 6 | Target Leakage Screen | Flag features with \|corr\| > 0.95 to target. If found: raise warning, do NOT silently include | Warning or "No leakage detected" |
| 7 | Available Tables | List all tables in same schema with row counts — helps user identify feature sources for next notebook | Table of available sources |

**MLOps best practice:** EDA is NOT optional. Skipping it leads to leakage, wrong metrics, and failed production models.

---

### 4.2 — 02_feature_engineering.py (User-Editable Template)

| Cell # | Title | What it does |
| --- | --- | --- |
| 1 | Setup & Config | Load config, init Spark, print entity_key + timestamp_key |
| 2 | Source Tables (USER EDITS) | Clearly marked section where user defines which tables to read and what features to compute. Rich sample code included (see below) |
| 3 | Compute Features | User’s feature logic executes here (aggregations, windows, joins) |
| 4 | Write Feature Table | Write computed features to `feature_table_name` in Unity Catalog |
| 5 | Validate | Check: table created, no nulls in entity_key, row count > 0, print schema |

**Sample code included in Cell 2 (user edits this):**
```python
# ══════════════════════════════════════════════════════════════
# SAMPLE 1: Aggregation features
# ══════════════════════════════════════════════════════════════
# features_txn = spark.sql("""
#     SELECT customer_id,
#            SUM(amount) as total_spend_30d,
#            COUNT(*) as txn_count_30d,
#            AVG(amount) as avg_txn_amount,
#            MAX(amount) as max_txn_amount,
#            DATEDIFF(current_date(), MAX(txn_date)) as days_since_last_txn
#     FROM catalog.schema.transactions
#     WHERE txn_date >= date_sub(current_date(), 30)
#     GROUP BY customer_id
# """)

# ══════════════════════════════════════════════════════════════
# SAMPLE 2: Window function features
# ══════════════════════════════════════════════════════════════
# features_window = spark.sql("""
#     SELECT customer_id, event_timestamp,
#            LAG(amount, 1) OVER (PARTITION BY customer_id ORDER BY txn_date) as prev_txn_amount,
#            amount - AVG(amount) OVER (PARTITION BY customer_id) as amount_vs_avg,
#            ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY txn_date) as txn_sequence
#     FROM catalog.schema.transactions
# """)

# ══════════════════════════════════════════════════════════════
# SAMPLE 3: Lookup / passthrough features
# ══════════════════════════════════════════════════════════════
# features_profile = spark.sql("""
#     SELECT customer_id,
#            DATEDIFF(current_date(), signup_date) as tenure_days,
#            account_type,
#            region,
#            credit_score
#     FROM catalog.schema.customer_profile
# """)

# ══════════════════════════════════════════════════════════════
# SAMPLE 4: Join multiple sources
# ══════════════════════════════════════════════════════════════
# final_features = features_txn.join(features_profile, on="customer_id", how="left")
```

**MLOps best practice:** Feature engineering is done in Spark (scales to millions of rows). The output feature table is what the FE Client references during training and scoring.

---

### 4.3 — 03_train_tune.py (Training + Hyperparameter Tuning)

| Cell # | Title | What it does |
| --- | --- | --- |
| 1 | Setup & Config | Load config, install optuna/lightgbm if needed, `dbutils.library.restartPython()` |
| 2 | MLflow Experiment | Set experiment, log environment metadata (DBR version, package versions, seeds) |
| 3 | Create Training Set | `fe.create_training_set(entity_df, feature_lookups=[...])` → point-in-time join |
| 4 | Convert to Pandas + Split | `.load_df().toPandas()` → train/val/test split. **Lock test away** (never used in this notebook) |
| 5 | Define Preprocessing Pipeline | `ColumnTransformer` with `StandardScaler` (numeric) + `OneHotEncoder` (categorical) |
| 6 | Baselines | DummyClassifier + LogisticRegression (or DummyRegressor + Ridge). Log params + metrics |
| 7 | Optuna Objective Function | Define objective: build Pipeline(preprocessor, model), fit, score val set, return metric |
| 8 | Run Parallel HPO | `study.optimize(objective, n_trials=N, n_jobs=parallel_trials)`. Nested MLflow runs per trial |
| 9 | Best Model | Load best trial params, retrain on train set, log full artifact with `fe.log_model()` |
| 10 | Summary | Print best run_id, model URI, val metrics. Save test split for next notebook |

**Hyperparameter ranges (per algorithm):**

| Algorithm | Classification Objective | Regression Objective |
| --- | --- | --- |
| LightGBM | val_f1 (or val_pr_auc if imbalanced) | val_rmse (minimize) |
| XGBoost | val_f1 (or val_pr_auc if imbalanced) | val_rmse (minimize) |
| Random Forest | val_f1 (or val_pr_auc if imbalanced) | val_rmse (minimize) |

**Parallel tuning strategy:**
- `n_jobs` in `study.optimize()` for driver-level parallelism
- Each trial’s model gets `n_jobs = total_cores // parallel_trials` (prevent CPU thrashing)
- TPE sampler with `seed=42` for reproducibility
- `MedianPruner(n_warmup_steps=10)` for early stopping of bad trials

**MLflow logging per trial:**
- Params: all hyperparameters + `trial_number`
- Metrics: val_f1, val_precision, val_recall, val_auc, val_pr_auc, fit_time_seconds
- Tags: `stage="tuning"`, `pruned=true/false`, `best_trial=true/false`
- Artifact: ONLY best trial logs full model (saves storage)

---

### 4.4 — 04_evaluate.py (Holdout Evaluation)

| Cell # | Title | What it does |
| --- | --- | --- |
| 1 | Setup & Config | Load config, load model from best run_id URI |
| 2 | Audit Training Run | Load parent + nested run metadata. Compare train vs val metrics. Flag overfitting (gap > threshold) |
| 3 | Schema Check | Verify test features match model signature. Report mismatches before predicting |
| 4 | Score Holdout | `fe.score_batch(model_uri, test_df)` on locked test split |
| 5 | Compute Metrics | Classification: precision, recall, F1, AUC, PR-AUC. Regression: MAE, RMSE, R². Log all with `test_` prefix |
| 6 | Diagnostic Plots | Confusion matrix, ROC, PR curve (classification) or residual plot (regression). Log as MLflow artifacts |
| 7 | Summary | Narrative: training health + test performance + recommendation (pass/fail) |

**Hard rules:**
- NEVER change hyperparameters based on test metrics
- NEVER re-tune if test looks bad (that’s a new training episode)
- Test set is used ONCE, results logged, done

---

### 4.5 — 05_register.py (UC Registration + Aliases)

| Cell # | Title | What it does |
| --- | --- | --- |
| 1 | Setup & Config | Load config, init MlflowClient |
| 2 | Compare Runs | Pull top runs from experiment, rank by val metric, show comparison table |
| 3 | Smoke Test | Load model from `runs:/<run_id>/model`, predict on sample data, verify output |
| 4 | Register to UC | `mlflow.register_model(model_uri, "catalog.schema.model_name")` |
| 5 | Set Aliases | `client.set_registered_model_alias(..., "Champion", version)`. Set Challenger if previous version exists |
| 6 | Dependency Capture | Log pip_requirements, conda_env, DBR version, key package versions as model tags |
| 7 | Deployment Log | Append row to `DEPLOYMENT_LOG.md`: timestamp, user, event, model FQN, version, alias, links, dependencies |

## 4A. Deep Learning Notebook Specification — 03b_train_deep_learning.py

### When Generated
Only included when `train.model_algorithm` is `"deep_learning"` or `"all"`. When `"all"` is selected, this notebook runs IN ADDITION to `03_train_tune.py` — both produce a best model, and `05_register.py` compares them to pick the overall winner.

### Compute Requirement
When `train.deep_learning.compute: "gpu"`, the framework switches the job task to **GPU serverless** (Azure NC-series). The notebook auto-detects GPU availability and falls back to CPU with a warning.

---

### Notebook Cell Specification

| Cell # | Title | What it does |
| --- | --- | --- |
| 0 | Header Markdown | Purpose, prerequisites, inputs/outputs, estimated runtime (10-30 min with GPU) |
| 1 | Setup & Config | Load config, detect GPU (`torch.cuda.is_available()`), install pytorch/tensorflow if needed, restart kernel |
| 2 | Section: Data Preparation | Explains why DL needs different prep than tree models |
| 3 | Load Training Set | `fe.create_training_set()` → `.toPandas()`. Separate numeric vs categorical. Apply StandardScaler to numeric (required for neural nets). Encode categoricals as integers for embedding layers |
| 4 | Section: Network Architecture | Explains architecture choices |
| 5 | Define Model Architecture | Build model from config (`tabular_mlp` / `tabular_resnet` / `custom`). Print model summary (param count, layer shapes) |
| 6 | Section: Hyperparameter Tuning | Explains what Optuna tunes for DL |
| 7 | Optuna HPO for DL | Tune: learning_rate, weight_decay, dropout, batch_size, hidden_layer_sizes. Each trial trains with early stopping. Log val metric per trial |
| 8 | Section: Train Best Model | Explains final training |
| 9 | Train Final Model | Retrain with best hyperparams on full train set. Plot training curves (loss + metric vs epoch). Save checkpoint |
| 10 | Wrap in Pyfunc | Create `DeepLearningModel(mlflow.pyfunc.PythonModel)` that includes preprocessing (scaling + encoding) inside `predict()` |
| 11 | Log with FE Client | `fe.log_model(pyfunc_wrapper, training_set=..., flavor=mlflow.pyfunc)` |
| 12 | Summary | Print run_id, val metric, model size, training time |

---

### Deep Learning Hyperparameter Search Space

| Hyperparameter | Range | Scale | Notes |
| --- | --- | --- | --- |
| `learning_rate` | [0.0001, 0.01] | log-uniform | Lower than tree models |
| `weight_decay` | [1e-5, 0.01] | log-uniform | L2 regularization |
| `dropout` | [0.1, 0.5] | uniform | Between hidden layers |
| `batch_size` | [64, 512] | categorical: [64, 128, 256, 512] | Affects generalization |
| `hidden_layers` | varies | categorical configurations | See below |
| `activation` | ["relu", "gelu", "silu"] | categorical | |
| `optimizer` | ["adam", "adamw"] | categorical | AdamW preferred for weight decay |

**Hidden layer configurations searched:**
```python
layer_configs = [
    [128, 64],
    [256, 128],
    [256, 128, 64],
    [512, 256, 128],
    [256, 128, 64, 32],
]
```

---

### Predefined Architectures (nn_architectures.py)

**tabular_mlp (default):**
```python
class TabularMLP(nn.Module):
    """Standard feedforward network with batch norm + dropout."""
    def __init__(self, input_dim, hidden_layers, output_dim, dropout, activation):
        # Input → [Linear → BatchNorm → Activation → Dropout] × N → Output
```

**tabular_resnet:**
```python
class TabularResNet(nn.Module):
    """Residual network for tabular data (skip connections improve gradient flow)."""
    def __init__(self, input_dim, hidden_dim, n_blocks, output_dim, dropout):
        # Input → Linear → [ResBlock × N] → Linear → Output
        # ResBlock: Linear → BatchNorm → Activation → Dropout → Linear → + skip
```

**custom (user template):**
```python
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  🔧 USER ACTION REQUIRED — Define your custom architecture         ║
# ╠══════════════════════════════════════════════════════════════════════╣
# ║  Your model must:                                                    ║
# ║  • Accept input_dim (number of features after preprocessing)        ║
# ║  • Return output_dim (1 for regression, n_classes for classification)║
# ║  • Be a nn.Module subclass with a forward() method                  ║
# ╚══════════════════════════════════════════════════════════════════════╝
class CustomModel(nn.Module):
    def __init__(self, input_dim, output_dim, **kwargs):
        super().__init__()
        # YOUR ARCHITECTURE HERE
```

---

### DL-Specific MLOps Considerations

| Concern | How framework handles it |
| --- | --- |
| Preprocessing sync (train ↔ inference) | Scaling + encoding baked into pyfunc `predict()` — same as migrate mode pattern |
| GPU at training, CPU at serving | Model trains on GPU, but pyfunc wraps as CPU-compatible. Serving endpoint uses CPU by default (DL tabular models are fast on CPU at single-sample inference) |
| Large model artifacts | PyTorch state_dict (~5-50MB for tabular). Logged as `context.artifacts["model_state"]` |
| Reproducibility | `torch.manual_seed()`, `torch.cuda.manual_seed_all()`, `torch.backends.cudnn.deterministic=True` |
| Early stopping | `patience` from config. Restores best checkpoint (lowest val loss) |
| Training curves | Plotted and logged as MLflow artifact (`training_curves.png`) |

---

### When `model_algorithm: "all"` Includes Deep Learning

The training pipeline DAG becomes:
```
02_features → 03_train_tune (tree models) ─────────┐
           ↘ 03b_train_deep_learning (neural net) ──┤→ 04_evaluate → 05_register
                                                     │
                                    (05_register picks overall best
                                     from both notebooks' best models)
```

`05_register.py` loads BOTH best models, compares val metrics, and registers the winner as Champion.

---
---

## 4B. Model Explainability Notebook Specification — 04b_explainability.py

### When Generated
Always included when `explainability.enabled: true` in config (default: true). Runs AFTER `04_evaluate.py` and BEFORE `05_register.py` in the pipeline DAG.

### Purpose
Provide interpretable explanations of the model's predictions: which features matter most, how they influence predictions, and visual evidence for stakeholder communication.

---

### Notebook Cell Specification

| Cell # | Title | What it does |
| --- | --- | --- |
| 0 | Header Markdown | Purpose: "Explain model behavior. Which features drive predictions? How does each feature influence the outcome?" Prerequisites: 04_evaluate must pass |
| 1 | Setup & Config | Load config, load best model from MLflow, load test data subset |
| 2 | Section: SHAP Values | Explains SHAP: "Assigns each feature a contribution score for every prediction. Positive = pushes toward positive class. Negative = pushes toward negative class" |
| 3 | Compute SHAP | `shap.TreeExplainer` (for tree models) or `shap.DeepExplainer` / `shap.KernelExplainer` (for DL/pyfunc). Subsample to `n_samples` from config |
| 4 | SHAP Summary Plot | Beeswarm plot: global feature importance with directionality. Log as MLflow artifact |
| 5 | SHAP Bar Plot | Mean absolute SHAP values — simpler ranking. Log as MLflow artifact |
| 6 | SHAP Dependence Plots | Top-3 features: how feature value maps to SHAP value (shows non-linear effects). Log as MLflow artifacts |
| 7 | Section: Permutation Importance | Explains: "Shuffle each feature and measure performance drop. Larger drop = more important" |
| 8 | Permutation Importance | `sklearn.inspection.permutation_importance()` on test set. Bar chart of feature importances with std bars |
| 9 | Section: Partial Dependence | Explains: "Shows the marginal effect of a feature on predictions, averaging out other features" |
| 10 | Partial Dependence Plots | `sklearn.inspection.PartialDependenceDisplay` for top-5 features. Grid of PDP curves |
| 11 | Section: Local Explanations | Explains: "Why did the model make THIS specific prediction?" |
| 12 | Local SHAP (Waterfall) | Pick 3 representative samples (1 high-confidence positive, 1 high-confidence negative, 1 borderline). Waterfall plot for each showing feature contributions |
| 13 | Summary Table | Feature ranking table: Feature Name → SHAP Importance → Permutation Importance → Agreement? |
| 14 | Log All to MLflow | Log all plots as artifacts under `explainability/` folder in the run. Log top-10 features as MLflow params |
| 15 | Summary Markdown | "Top predictive features: X, Y, Z. Non-linear effects detected in A, B. → Next: 05_register.py" |

---

### Explainability Method Selection by Model Type

| Model Type | SHAP Explainer | Speed | Notes |
| --- | --- | --- | --- |
| LightGBM / XGBoost | `shap.TreeExplainer` | Fast (exact) | Native support, no approximation needed |
| Random Forest | `shap.TreeExplainer` | Fast (exact) | Same as boosted trees |
| Deep Learning (pyfunc) | `shap.DeepExplainer` | Medium | Requires model internals; fallback to KernelExplainer |
| Migrated Model (pyfunc) | `shap.KernelExplainer` | Slow | Model-agnostic; uses `n_samples` subsampling heavily |

**Fallback logic:**
```python
if model_type in ["lightgbm", "xgboost", "random_forest"]:
    explainer = shap.TreeExplainer(model)
elif model_type == "deep_learning" and framework == "pytorch":
    try:
        explainer = shap.DeepExplainer(model, background_data)
    except:
        explainer = shap.KernelExplainer(model.predict, background_data)
else:
    # Migrated or unknown model type — model-agnostic
    explainer = shap.KernelExplainer(model.predict, shap.sample(X_test, 100))
```

---

### Explainability Outputs (logged to MLflow)

```
runs:/{run_id}/artifacts/
└── explainability/
    ├── shap_summary_beeswarm.png      ← Global feature importance + direction
    ├── shap_bar_importance.png        ← Ranked feature importance (simplified)
    ├── shap_dependence_feature1.png   ← Non-linear effects
    ├── shap_dependence_feature2.png
    ├── shap_dependence_feature3.png
    ├── shap_waterfall_positive.png    ← Why this sample = positive
    ├── shap_waterfall_negative.png    ← Why this sample = negative
    ├── shap_waterfall_borderline.png  ← Why this sample is uncertain
    ├── permutation_importance.png     ← Alternative importance method
    ├── partial_dependence_grid.png    ← PDP for top-5 features
    └── feature_ranking_table.csv      ← Exportable importance table
```

---

### MLOps Value of Explainability

| Stakeholder | What they get from this notebook |
| --- | --- |
| **Data Scientist** | Validate model learned sensible patterns (not spurious correlations) |
| **Business Stakeholder** | "The model predicts churn because of X, Y, Z" — actionable insights |
| **Compliance / Risk** | Evidence that model decisions are explainable and non-discriminatory |
| **Model Reviewer** | Artifact-backed proof of feature importance before promoting to Champion |

---

### Updated Training Pipeline DAG (with DL + Explainability)

```
01_eda → 02_features ─┬→ 03_train_tune ──────────┬→ 04_evaluate → 04b_explainability → 05_register
                       └→ 03b_deep_learning ──────┘
                          (if DL enabled)                (if explainability.enabled)
```

**Conditional generation:**
- `03b_train_deep_learning.py` → only if `model_algorithm` includes deep learning
- `04b_explainability.py` → only if `explainability.enabled: true`
- Both are optional tasks in the DAB job definition (task runs based on config)

## 5. Notebook Specifications — MIGRATE Mode

### 5.1 — 01_validate_model.py (Load + Inspect + Smoke Test)

| Cell # | Title | What it does |
| --- | --- | --- |
| 1 | Setup & Config | Load config, read `migrate.model_path` from Volume |
| 2 | Load Pickle | `joblib.load()` from Volume. Print type, class, `get_params()` if available |
| 3 | Inspect Model | Print input expectations: n_features, feature_names (if stored), expected dtypes |
| 4 | Smoke Test | Feed random/zero data of correct shape → verify `.predict()` runs without error |
| 5 | Signature Inference | If reference_table available: infer signature from sample input/output. Otherwise: manual definition |
| 6 | Reference Validation | Score reference_table, compare predictions vs `source_system_prediction` column. Assert match_rate >= 99% |

**Key principle:** This notebook answers: "Does the pickle work in Databricks? Does it produce the same outputs as the source system?"

---

### 5.2 — 02_feature_engineering.py (Same Template as Train Mode)

Identical to train mode. User defines feature computation logic here. Same samples provided.

---

### 5.3 — 03_wrap_and_register.py (Pyfunc Wrapper + Registration)

| Cell # | Title | What it does |
| --- | --- | --- |
| 1 | Setup & Config | Load config, init FE Client, load raw model pickle |
| 2 | Define Preprocessing (USER EDITS) | User defines preprocessing inside `predict()` method. Rich samples provided |
| 3 | Create Pyfunc Wrapper | `MigratedModel(mlflow.pyfunc.PythonModel)` class with preprocessing + model |
| 4 | Validate Wrapper | Score sample data through the wrapper, verify output matches reference |
| 5 | Log with FE Client | `fe.log_model(wrapper, training_set=..., artifacts={...})` → embeds feature lookups |
| 6 | Register to UC | Register model, set Champion alias |
| 7 | Deployment Log | Append migration event to DEPLOYMENT_LOG.md |

**The Pyfunc wrapper pattern (Cell 2-3):**
```python
class MigratedModel(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        self.model = joblib.load(context.artifacts["model"])
        # Load any additional preprocessing artifacts
        # self.scaler = joblib.load(context.artifacts["scaler"])
    
    def predict(self, context, model_input, params=None):
        processed = model_input.copy()
        
        # ╔══════════════════════════════════════════════════════════╗
        # ║  YOUR PREPROCESSING HERE                              ║
        # ╠══════════════════════════════════════════════════════════╣
        # ║  This MUST replicate the source system's transforms  ║
        # ║  Examples:                                            ║
        # ║  - Numeric scaling (StandardScaler / MinMaxScaler)    ║
        # ║  - Categorical encoding (OneHot / Label / Target)     ║
        # ║  - Missing value imputation                           ║
        # ║  - Feature selection / dropping columns               ║
        # ║  - Custom transformations (log, binning, etc.)        ║
        # ╚══════════════════════════════════════════════════════════╝
        
        return self.model.predict(processed)
```

**Why preprocessing is ALWAYS explicit:**
- Auditable: anyone can read the notebook and see exactly what transforms run
- Debuggable: if predictions drift, you can inspect each step
- Portable: same transforms apply in batch inference AND serving (baked into artifact)
- Compliant: model signature enforces input schema — wrong features fail loudly

## 6. Notebook Specifications — COMMON (Both Modes)

### 6.1 — 06_batch_inference.py (Recurring Batch Scoring)

| Cell # | Title | What it does |
| --- | --- | --- |
| 1 | Setup & Config | Load config, init FE Client |
| 2 | Load Scoring Set | `spark.table(batch.scoring_table)` — entities to score (just entity_key + timestamp) |
| 3 | Score with FE Client | `fe.score_batch(model_uri="models:/...@Champion", df=scoring_df)` — auto-fetches features, applies preprocessing, predicts |
| 4 | Add Metadata | Add columns: `prediction_timestamp`, `model_version`, `model_id` |
| 5 | Write Predictions | Append to `batch.output_table` (always append for monitoring history) |
| 6 | Log Run | Append to DEPLOYMENT_LOG.md: rows scored, model version, timestamp |

**How fe.score_batch() ensures sync:**
1. Reads feature lookup specs embedded in model artifact
2. Auto-joins features from feature tables (Spark)
3. Converts to pandas internally
4. Runs the full sklearn Pipeline (preprocessing + predict) OR pyfunc.predict()
5. Returns Spark DataFrame with predictions

**User NEVER manually replicates preprocessing logic here.** It’s inside the artifact.

---

### 6.2 — 07_serve.py (Real-Time Serving Endpoint)

| Cell # | Title | What it does |
| --- | --- | --- |
| 1 | Setup & Config | Load config, init WorkspaceClient, resolve Champion version from alias |
| 2 | Create Endpoint | `w.serving_endpoints.create_and_wait()` with CPU/Small/scale-to-zero + AI Gateway inference table enabled |
| 3 | Test Endpoint | Send sample request via SDK, print prediction |
| 4 | Deployment Log | Append: endpoint name, model URI, inference table location |

**Serving + Feature Lookup:**
- When model was logged with `fe.log_model()`, the endpoint auto-fetches features from feature tables
- For online/low-latency: publish feature table as Online Table (optional enhancement)
- Preprocessing runs inside the model artifact (same as batch)

---

### 6.3 — 08_monitor.py (Lakehouse Monitoring)

| Cell # | Title | What it does |
| --- | --- | --- |
| 1 | Setup & Config | Load config, determine which inference table to monitor |
| 2 | Validate Table | Check table exists, timestamp is TimestampType, required columns present |
| 3 | Create Monitor | `quality_monitors.create()` with InferenceLog profile (classification or regression) |
| 4 | Configure Schedule | Add `MonitorCronSchedule` if `monitoring.refresh_schedule` is set |
| 5 | Trigger Initial Refresh | `quality_monitors.run_refresh()` |
| 6 | Display Metrics | Show profile_metrics and drift_metrics tables |

**Monitor column mapping (auto-inferred):**
- `prediction_col` → "prediction"
- `timestamp_col` → "prediction_timestamp" (batch) or "timestamp" (serving processed)
- `model_id_col` → "model_id"
- `label_col` → label_column from config (if ground truth available)

**Monitors BOTH inference paths:**
- Batch predictions table (`batch.output_table`)
- Processed serving inference table (`{endpoint}_inference_processed`)

---

### 6.4 — 09_process_inference.py (Flatten Serving Payload)

| Cell # | Title | What it does |
| --- | --- | --- |
| 1 | Setup & Config | Load config, read model signature for schema |
| 2 | Build Parse Schema | From model signature → StructType for request/response JSON parsing |
| 3 | Flatten Payload | `timestamp_ms/1000` → timestamp. Parse request JSON → individual feature columns. Parse response → prediction column |
| 4 | Write Processed Table | Append to `{endpoint}_inference_processed` |

**Only generated if `inference_mode` includes "serving".** This notebook is scheduled as a separate recurring job to continuously process new serving payloads into a flat table for monitoring.

---

### 6.5 — helpers.py (Shared Utilities)

Not a notebook — a Python module imported by all notebooks:

```python
# Key functions:
def load_config(config_path=None) -> dict
    """Load config from job params OR widget OR file fallback"""

def get_current_user() -> str
    """Get current user email for logging"""

def append_deployment_log(event, resource, notes, log_path=None)
    """Append a row to DEPLOYMENT_LOG.md"""

def get_feature_lookups(config) -> list
    """Build FeatureLookup list from config"""

def detect_class_imbalance(y, threshold=5.0) -> bool
    """Check if class ratio exceeds threshold"""
```

## 7. DAB Configuration (Declarative Automation Bundles)

### 7.1 — databricks.yml (Main Config)

```yaml
bundle:
  name: mlops-framework

include:
  - resources/*.yml

variables:
  catalog:
    default: "dev_catalog"
  schema:
    default: "dev_schema"
  model_name:
    default: "churn_model"
  warehouse_id:
    lookup:
      warehouse: "Shared SQL Warehouse"

targets:
  dev:
    default: true
    mode: development
    workspace:
      host: https://adb-xxxxx.azuredatabricks.net
    variables:
      catalog: "dev_catalog"
      schema: "dev_schema"

  staging:
    workspace:
      host: https://adb-yyyyy.azuredatabricks.net
    variables:
      catalog: "staging_catalog"
      schema: "staging_schema"

  prod:
    mode: production
    workspace:
      host: https://adb-zzzzz.azuredatabricks.net
    variables:
      catalog: "prod_catalog"
      schema: "prod_schema"
```

### 7.2 — resources/training_pipeline_job.yml

```yaml
resources:
  jobs:
    training_pipeline:
      name: "[${bundle.target}] MLOps Training Pipeline - ${var.model_name}"
      
      parameters:
        - name: config_path
          default: "${workspace.file_path}/config.yaml"
        - name: target
          default: "${bundle.target}"
      
      tasks:
        - task_key: "eda"
          notebook_task:
            notebook_path: ../src/notebooks/01_eda.py
            base_parameters:
              config_path: "{{job.parameters.config_path}}"

        - task_key: "feature_engineering"
          depends_on:
            - task_key: "eda"
          notebook_task:
            notebook_path: ../src/notebooks/02_feature_engineering.py
            base_parameters:
              config_path: "{{job.parameters.config_path}}"

        - task_key: "train_tune"
          depends_on:
            - task_key: "feature_engineering"
          notebook_task:
            notebook_path: ../src/notebooks/03_train_tune.py
            base_parameters:
              config_path: "{{job.parameters.config_path}}"

        - task_key: "train_deep_learning"    # Only if DL enabled in config
          depends_on:
            - task_key: "feature_engineering"
          run_if: ALL_SUCCESS
          notebook_task:
            notebook_path: ../src/notebooks/03b_train_deep_learning.py
            base_parameters:
              config_path: "{{job.parameters.config_path}}"

        - task_key: "evaluate"
          depends_on:
            - task_key: "train_tune"
            - task_key: "train_deep_learning"
          run_if: AT_LEAST_ONE_SUCCESS    # Proceeds if tree OR DL succeeds
          notebook_task:
            notebook_path: ../src/notebooks/04_evaluate.py
            base_parameters:
              config_path: "{{job.parameters.config_path}}"

        - task_key: "explainability"         # Only if explainability.enabled
          depends_on:
            - task_key: "evaluate"
          notebook_task:
            notebook_path: ../src/notebooks/04b_explainability.py
            base_parameters:
              config_path: "{{job.parameters.config_path}}"

        - task_key: "register"
          depends_on:
            - task_key: "explainability"
          run_if: ALL_DONE    # Proceeds even if explainability is skipped
          notebook_task:
            notebook_path: ../src/notebooks/05_register.py
            base_parameters:
              config_path: "{{job.parameters.config_path}}"

        - task_key: "serve"
          depends_on:
            - task_key: "register"
          run_if: ALL_SUCCESS
          notebook_task:
            notebook_path: ../src/notebooks/07_serve.py
            base_parameters:
              config_path: "{{job.parameters.config_path}}"

        - task_key: "monitor"
          depends_on:
            - task_key: "serve"
          run_if: ALL_DONE
          notebook_task:
            notebook_path: ../src/notebooks/08_monitor.py
            base_parameters:
              config_path: "{{job.parameters.config_path}}"

      permissions:
        - level: CAN_VIEW
          group_name: "users"
        - level: CAN_MANAGE_RUN
          group_name: "data-scientists"
```

### 7.3 — resources/batch_inference_job.yml

```yaml
resources:
  jobs:
    batch_inference:
      name: "[${bundle.target}] Batch Inference - ${var.model_name}"
      
      schedule:
        quartz_cron_expression: "0 0 6 * * ? *"    # Daily 6am
        timezone_id: "UTC"
      
      tasks:
        - task_key: "score"
          notebook_task:
            notebook_path: ../src/notebooks/06_batch_inference.py
            base_parameters:
              config_path: "{{job.parameters.config_path}}"

        - task_key: "refresh_monitor"
          depends_on:
            - task_key: "score"
          notebook_task:
            notebook_path: ../src/notebooks/08_monitor.py
            base_parameters:
              config_path: "{{job.parameters.config_path}}"
              action: "refresh_only"   # skip creation, just refresh

      permissions:
        - level: CAN_VIEW
          group_name: "users"
```

### 7.4 — Job DAG Visualization

**Training Pipeline (one-shot, run manually or on model refresh):**
```
                          ┌→ 03_train_tune (tree models) ─────┐
01_eda → 02_features ───┤                                      ├→ 04_evaluate → 04b_explain → 05_register → 07_serve → 08_monitor
                          └→ 03b_deep_learning (DL model) ──┘
                             (parallel, if DL enabled)         (if explainability.enabled)
```

**Batch Inference (recurring, scheduled):**
```
06_batch_inference ──→ 08_monitor (refresh only)
```

**Serving Payload Processing (recurring, if serving enabled):**
```
09_process_inference ──→ 08_monitor (refresh only)
```

## 8. MLflow Logging Strategy

### 8.1 — Experiment Structure

```
📁 Experiment: /Users/{user}/{model_name}_experiment
│
├── 🏃 "baseline_dummy"           ← params + metrics only
├── 🏃 "baseline_logistic"        ← params + metrics only
├── 🏃 "optuna_tuning" (parent)
│   ├── 🏃 "trial_0"              ← params + metrics (no artifact)
│   ├── 🏃 "trial_1"              ← params + metrics (no artifact)
│   ├── ...                        ← (all trials)
│   └── 🏃 "trial_37" ⭐ BEST      ← params + metrics + FULL ARTIFACT
└── 🏃 "holdout_evaluation"       ← test metrics + diagnostic plots
```

### 8.2 — What Gets Logged

| Category | Every Trial | Best Trial Only | Holdout Eval |
| --- | --- | --- | --- |
| **Hyperparameters** | ✅ All params | ✅ | ✅ model_uri, test_size |
| **Validation Metrics** | ✅ val_f1, val_auc, etc. | ✅ | ❌ |
| **Test Metrics** | ❌ | ❌ | ✅ test_f1, test_auc, etc. |
| **Model Artifact** (.pkl) | ❌ | ✅ Full Pipeline | ❌ (uses existing) |
| **Signature + input_example** | ❌ | ✅ | ❌ |
| **pip_requirements** | ❌ | ✅ | ❌ |
| **Diagnostic Plots** | ❌ | ❌ | ✅ confusion, ROC, PR |
| **fit_time_seconds** | ✅ | ✅ | ❌ |
| **Tags** (stage, trial#) | ✅ | ✅ | ✅ |

### 8.3 — Metrics Logged

**Classification (per trial + holdout):**
```
val_f1, val_precision, val_recall, val_auc, val_pr_auc, val_accuracy, fit_time_seconds
test_f1, test_precision, test_recall, test_auc, test_pr_auc, test_accuracy, train_val_gap_f1
```

**Regression (per trial + holdout):**
```
val_rmse, val_mae, val_r2, val_mape, fit_time_seconds
test_rmse, test_mae, test_r2, test_mape, train_val_gap_r2
```

### 8.4 — Environment Params (logged once on best trial)
```
dbr_version, python_version, mlflow_version, sklearn_version,
lightgbm_version/xgboost_version, optuna_version,
training_table, feature_table, n_train_rows, n_val_rows,
n_test_rows, n_features, entity_key, label_column, random_seed
```

### 8.5 — Why Only Best Trial Gets Artifact
- 50 trials × model artifact = GB of storage nobody uses
- Params + metrics for ALL trials enable comparison in MLflow UI
- If user wants to reproduce trial #12 later: params are logged, just retrain
- Trade-off: save storage, maintain full comparison capability

## 9. Feature Engineering Client — The Lineage Backbone

### 9.1 — How FE Client Threads Through the Framework

```
02_feature_engineering.py        03_train_tune.py             05_register.py / 03_wrap_and_register.py
┌───────────────────────┐    ┌───────────────────────┐    ┌───────────────────────┐
│ User writes Spark SQL/  │    │ fe.create_training_set()│    │ fe.log_model()          │
│ PySpark to compute     │    │ joins features to       │    │ embeds feature lookup   │
│ features → writes to   │────│ entity+label table      │────│ specs INTO the model    │
│ feature_table in UC    │    │ (point-in-time correct) │    │ artifact                │
└───────────────────────┘    └───────────────────────┘    └───────────────────────┘
                                                                       │
                              ┌─────────────────────────────────────────────┤
                              │                         MODEL ARTIFACT IN UC                   │
                              │  ┌───────────────┐  ┌───────────────┐  ┌───────────────┐  │
                              │  │ Preprocessing │  │ Feature       │  │ Model         │  │
                              │  │ Pipeline      │  │ Lookup Specs  │  │ (estimator)   │  │
                              │  └───────────────┘  └───────────────┘  └───────────────┘  │
                              └───────────────────┬────────────────┬────────────────┘
                                                   │                │
                              ┌───────────────────┴────────────────┴────────────────┐
                              │                                                                │
          ┌─────────────────┤                                                                ├────────────────┐
          │                 │                                                                │                │
          ▼                 ▼                                                                ▼                ▼
  06_batch_inference     07_serve                                                                             
  fe.score_batch()       Endpoint auto-                                                                       
  → auto-joins           fetches features                                                                     
  → preprocesses         → preprocesses                                                                       
  → predicts             → predicts                                                                           
```

### 9.2 — Key API Calls

| Notebook | FE Client Call | Purpose |
| --- | --- | --- |
| 03_train_tune | `fe.create_training_set(entity_df, feature_lookups, label)` | Point-in-time feature join for training |
| 05_register | `fe.log_model(model, training_set, flavor)` | Embed feature specs in artifact |
| 03_wrap_and_register | `fe.log_model(pyfunc_wrapper, training_set, flavor)` | Same — for migrated models |
| 06_batch_inference | `fe.score_batch(model_uri, scoring_df)` | Auto-join + preprocess + predict at scale |

### 9.3 — How Preprocessing Stays in Sync

| Scenario | Mechanism |
| --- | --- |
| Training vs batch inference | Preprocessing is INSIDE the sklearn Pipeline (train mode) or pyfunc.predict() (migrate mode). Same artifact runs everywhere |
| Training vs serving | Same artifact deployed to endpoint. Serving calls the same predict() |
| Feature schema changes | Model signature enforces schema. Incompatible changes fail loudly at score time |
| Feature values drift | Lakehouse Monitoring detects drift. Alert triggers retraining |

### 9.4 — The Spark ↔ Pandas Boundary (Critical Architecture)

```
┌────────────────────────────────┐    ┌────────────────────────────────┐
│      SPARK (scale)              │    │      PANDAS (model)              │
│                                │    │                                │
│  • Read tables (millions rows)  │    │  • sklearn Pipeline             │
│  • Feature engineering (SQL)    │    │  • LightGBM / XGBoost           │
│  • fe.create_training_set()     │────│  • .toPandas() for training     │
│  • fe.score_batch() at scale    │←───│  • StandardScaler + OneHot      │
│  • Write predictions to Delta   │    │  • model.fit(X_train, y_train)  │
└────────────────────────────────┘    └────────────────────────────────┘
```

**Spark** handles data at scale. **Pandas/sklearn** handles model building. **FE Client** bridges both worlds seamlessly.

## 10. Hyperparameter Tuning Specification

### 10.1 — Model Selection by Task Type

| Task | Baselines (fixed, no tuning) | Tunable Models | Default |
| --- | --- | --- | --- |
| Classification | DummyClassifier (most_frequent), LogisticRegression (C=1, balanced) | LightGBM, XGBoost, Random Forest, **Deep Learning (MLP/ResNet)** | LightGBM |
| Regression | DummyRegressor (mean), Ridge (alpha=1.0) | LightGBM, XGBoost, Random Forest, **Deep Learning (MLP/ResNet)** | LightGBM |

### 10.2 — LightGBM Search Space

| Hyperparameter | Range | Scale | Notes |
| --- | --- | --- | --- |
| learning_rate | [0.01, 0.3] | log-uniform | Low LR + more trees = better generalization |
| n_estimators | [100, 1000] step=50 | int | Coupled with learning_rate |
| max_depth | [3, 12] | int | Complexity control |
| num_leaves | [20, 150] | int | Primary LightGBM knob |
| min_child_samples | [5, 100] | int | Regularization |
| subsample | [0.5, 1.0] | uniform | Row sampling per tree |
| colsample_bytree | [0.4, 1.0] | uniform | Feature sampling per tree |
| reg_alpha | [1e-8, 10.0] | log-uniform | L1 regularization |
| reg_lambda | [1e-8, 10.0] | log-uniform | L2 regularization |
| min_split_gain | [0.0, 1.0] | uniform | Minimum gain to split |
| class_weight | ["balanced", None] | categorical | Classification only |

### 10.3 — XGBoost Search Space

| Hyperparameter | Range | Scale | Notes |
| --- | --- | --- | --- |
| learning_rate | [0.01, 0.3] | log-uniform | |
| n_estimators | [100, 1000] step=50 | int | |
| max_depth | [3, 12] | int | Depth-wise growth |
| min_child_weight | [1, 20] | int | Analogous to min_child_samples |
| gamma | [0.0, 5.0] | uniform | Min loss reduction for split |
| subsample | [0.5, 1.0] | uniform | |
| colsample_bytree | [0.4, 1.0] | uniform | |
| colsample_bylevel | [0.4, 1.0] | uniform | Extra per-depth sampling |
| reg_alpha | [1e-8, 10.0] | log-uniform | |
| reg_lambda | [1e-8, 10.0] | log-uniform | |
| scale_pos_weight | [1, class_ratio] | computed | Classification only |

### 10.4 — Random Forest Search Space

| Hyperparameter | Range | Scale | Notes |
| --- | --- | --- | --- |
| n_estimators | [100, 500] step=50 | int | Diminishing returns past 500 |
| max_depth | [3, 25, None] | int/None | None = grow until pure |
| min_samples_split | [2, 20] | int | |
| min_samples_leaf | [1, 20] | int | |
| max_features | ["sqrt", "log2", 0.5, 0.7] | categorical | |
| class_weight | ["balanced", "balanced_subsample", None] | categorical | Classification only |
| criterion | ["gini", "entropy"] / ["squared_error", "absolute_error"] | categorical | Task-dependent |

### 10.5 — Deep Learning Search Space (03b_train_deep_learning.py)

| Hyperparameter | Range | Scale | Notes |
| --- | --- | --- | --- |
| learning_rate | [0.0001, 0.01] | log-uniform | Lower than tree models |
| weight_decay | [1e-5, 0.01] | log-uniform | L2 regularization (AdamW) |
| dropout | [0.1, 0.5] | uniform | Between hidden layers |
| batch_size | [64, 128, 256, 512] | categorical | Affects generalization + speed |
| hidden_layers | see configs below | categorical | Architecture search |
| activation | ["relu", "gelu", "silu"] | categorical | Non-linearity |
| optimizer | ["adam", "adamw"] | categorical | AdamW preferred |
| n_epochs (max) | fixed from config | — | Early stopping controls actual epochs |

**Hidden layer configurations searched:**
```
[128, 64], [256, 128], [256, 128, 64], [512, 256, 128], [256, 128, 64, 32]
```

**DL-specific Optuna settings:**
- Sampler: TPESampler (same as tree models)
- Pruner: `optuna.pruners.PatientPruner(optuna.pruners.MedianPruner(), patience=5)` — DL needs more patience
- n_trials: 20-30 (DL trials are slower; each trains for up to `epochs` with early stopping)
- Parallel: `n_jobs=1` for DL (GPU can only run one model at a time)
- Early stopping: patience from config (default 10 epochs without improvement)

**Compute:** GPU serverless (auto-switched via config). Falls back to CPU with warning if GPU unavailable.

---

### 10.6 — Parallel Tuning Configuration

| Setting | Value | Rationale |
| --- | --- | --- |
| Sampler | TPESampler(seed=42) | Best for <200 trials |
| Pruner | MedianPruner(n_warmup_steps=10) | Kill bad trials early |
| n_trials | 50 (configurable) | Good balance for tabular |
| n_jobs (parallel) | auto: min(4, cores//2) | Sweet spot: 3x speed, minimal optimization loss |
| cores_per_trial | total_cores // n_jobs | Prevent CPU thrashing |

**Adaptive logic:**
```
IF n_train_rows < 10K:    → 5-fold CV, 30 trials, exclude Random Forest
IF 10K ≤ rows < 100K:     → single train/val split, 50 trials
IF rows ≥ 100K:            → single split, cap n_estimators at 500, 75 trials
IF class_imbalance > 5:1: → switch objective to val_pr_auc, enable class_weight
```

### 10.7 — Optuna Objective (classification)

| Condition | Objective | Direction |
| --- | --- | --- |
| Balanced classes | val_f1 | maximize |
| Imbalanced (ratio > 5:1) | val_pr_auc | maximize |
| Multiclass | val_f1_macro | maximize |

### 10.8 — Optuna Objective (regression)

| Condition | Objective | Direction |
| --- | --- | --- |
| Default | val_rmse | minimize |
| Outlier-prone data | val_mae | minimize |

## 11. MLOps Best Practices Baked Into the Framework

### 11.1 — Data Integrity & Leakage Prevention

| Practice | How Framework Enforces It |
| --- | --- |
| **Train/val/test isolation** | Test set created in 03_train_tune, NEVER touched until 04_evaluate. Physically separate variables |
| **Target leakage screening** | 01_eda auto-flags features with \|corr\| > 0.95 to target. Blocks pipeline if detected |
| **Point-in-time correctness** | `fe.create_training_set()` with `timestamp_lookup_key` prevents future data leaking into past predictions |
| **No test-based tuning** | Optuna objective uses ONLY val/CV metrics. Test metrics logged but never used for decisions |

### 11.2 — Reproducibility

| Practice | How Framework Enforces It |
| --- | --- |
| **Fixed seeds** | `random_seed=42` passed to all random components (split, sampler, model) |
| **Environment capture** | DBR version, Python version, all package versions logged as MLflow params |
| **Dependency pinning** | `pip_requirements` logged with model artifact. `conda_env` for full reproducibility |
| **Data versioning** | Training table name + timestamp logged. Feature table is a Unity Catalog managed table (versioned) |
| **Experiment tracking** | Every run, every trial, every metric recorded in MLflow |

### 11.3 — Model Governance & Lifecycle

| Practice | How Framework Enforces It |
| --- | --- |
| **Unity Catalog registration** | All models registered in UC with three-level namespace (catalog.schema.model) |
| **Champion/Challenger aliases** | Clear promotion path. Champion = production. Challenger = pending approval |
| **Deployment audit log** | Every registration, alias change, endpoint creation logged with timestamp + user + links |
| **Signature enforcement** | `input_example` + `signature` required at log time. Incompatible inputs fail at score time |
| **Model lineage** | FE Client embeds feature table references → full lineage from raw data to predictions |

### 11.4 — Inference Reliability

| Practice | How Framework Enforces It |
| --- | --- |
| **Preprocessing in artifact** | Never duplicated. Same Pipeline/pyfunc runs in training, batch, and serving |
| **Schema contract** | Model signature = schema contract. Feature table changes that break contract fail loudly |
| **Idempotent scoring** | Batch inference appends with dedup key. Re-runs don't produce duplicates |
| **Graceful degradation** | Serving endpoint scale-to-zero + health checks. Batch job handles empty tables gracefully |

### 11.5 — Monitoring & Observability

| Practice | How Framework Enforces It |
| --- | --- |
| **Data drift detection** | Lakehouse Monitoring with InferenceLog profile tracks feature distributions over time |
| **Prediction drift** | Monitor tracks prediction distribution shifts without needing ground truth |
| **Performance tracking** | When labels become available, monitor computes accuracy/F1/RMSE over time |
| **Scheduled refreshes** | Monitor auto-refreshes on schedule → continuous drift visibility |
| **Alerting (extensible)** | Framework generates monitor; user can add Databricks SQL Alerts on drift metrics |

### 11.6 — Multi-Environment Deployment

| Practice | How Framework Enforces It |
| --- | --- |
| **Environment isolation** | DAB targets (dev/staging/prod) use different catalogs + schemas. No cross-contamination |
| **Parameterized everything** | Catalog, schema, model name are variables — same code runs in any environment |
| **Promotion path** | dev → validate → staging → approve → prod. Each target has its own UC namespace |
| **Infrastructure as Code** | `databricks bundle deploy -t prod` reproduces entire pipeline in any workspace |
| **Permission separation** | DAB permissions: data-scientists get MANAGE_RUN in dev, only MANAGE in prod for admins |

### 11.7 — Operational Excellence

| Practice | How Framework Enforces It |
| --- | --- |
| **Config-driven** | Change config.yaml = new model. No code changes needed for a new use case |
| **Self-documenting** | Every notebook has clear cell titles, purpose comments, and output summaries |
| **Interactive fallback** | All notebooks work both as job tasks (params from job) AND standalone (widget fallback) |
| **Fail-fast** | Validation cells in every notebook. Bad data/missing tables caught early, not at prediction time |
| **Cost efficiency** | Scale-to-zero serving. Batch inference on serverless. No always-on compute |

### 11.8 — Security & Compliance

| Practice | How Framework Enforces It |
| --- | --- |
| **Unity Catalog governance** | All data, features, models, predictions in UC with ACLs |
| **No secrets in code** | All auth via workspace identity. No API keys or passwords in notebooks |
| **Audit trail** | DEPLOYMENT_LOG.md + MLflow lineage + UC lineage = full chain of custody |
| **Data residency** | Each target can point to a different workspace/region as needed |

## 12. INSTALL Notebook Specification

Following your cookie-cutter pattern from the Genie Multi-Agent Framework:

### Cell 1: Configuration
- Contains the full `config.yaml` content as a Python dict/YAML string
- THE ONLY CELL THE USER EDITS
- Well-commented with examples for each field
- Validates config schema before proceeding

### Cell 2: Generate Framework
- Reads config from Cell 1
- Generates all notebooks into `src/notebooks/` based on `mode`
- Generates `databricks.yml` + `resources/` for DABs
- Generates `deployment/DEPLOYMENT_LOG.md` template
- Generates `helpers.py` utility module
- Prints summary: what was created, what to edit next (02_feature_engineering)
- Does NOT deploy — just scaffolds

### Cell 3: Deploy (Optional)
- Runs `databricks bundle validate` first
- If valid: `databricks bundle deploy -t {target}`
- Creates the Lakeflow Jobs (training pipeline + batch inference)
- Prints: job URLs, next steps

### User Journey:
1. Clone/copy framework folder to their workspace
2. Open INSTALL.py → edit config (Cell 1)
3. Run Cell 2 → see "Generated 7 notebooks + DAB config"
4. Open `02_feature_engineering.py` → define features (using provided samples)
5. Optionally: open `03_wrap_and_register.py` (migrate mode) → define preprocessing
6. Run Cell 3 → `bundle deploy` creates jobs
7. Either: run full job end-to-end, OR open individual notebooks interactively
8. For a new model: change config → re-run Cell 2+3 → new instance deployed

## 13. Deployment Log Template (DEPLOYMENT_LOG.md)

```markdown
# Deployment Audit Log

> Auto-maintained by MLOps Framework. Each event appends a row.
> Links use format: [MLflow Run](https://<workspace>#mlflow/experiments/<exp_id>/runs/<run_id>)

| Timestamp (UTC) | User | Event | Resource | Version/Alias | Link | Dependencies | Notes |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 2026-06-15 14:30:00 | sunny.singh | model_trained | catalog.schema.churn_model | run_id=abc123 | [Run](link) | lgbm=4.5, sklearn=1.5 | val_f1=0.89 |
| 2026-06-15 14:35:00 | sunny.singh | holdout_evaluated | catalog.schema.churn_model | run_id=abc123 | [Run](link) | - | test_f1=0.87 |
| 2026-06-15 14:40:00 | sunny.singh | uc_registered | catalog.schema.churn_model | v1 | [Model](link) | lgbm=4.5, sklearn=1.5 | Champion |
| 2026-06-15 14:41:00 | sunny.singh | alias_set | catalog.schema.churn_model | Champion=v1 | [Model](link) | - | First version |
| 2026-06-15 14:50:00 | sunny.singh | endpoint_created | churn_model_serving | CPU/Small/s2z | [Endpoint](link) | - | Inference table enabled |
| 2026-06-15 14:55:00 | sunny.singh | monitor_created | catalog.schema.churn_predictions | InferenceLog | [Monitor](link) | - | daily refresh |
| 2026-06-16 06:00:00 | scheduled_job | batch_scored | catalog.schema.churn_predictions | v1/Champion | - | - | 45,000 rows scored |
```

**Events tracked:**
- `model_trained` — best trial logged
- `holdout_evaluated` — test metrics computed
- `uc_registered` — model registered to Unity Catalog
- `alias_set` — Champion/Challenger alias changed
- `endpoint_created` / `endpoint_updated` — serving endpoint lifecycle
- `monitor_created` / `monitor_refreshed` — monitoring lifecycle
- `batch_scored` — recurring batch inference completed
- `model_migrated` — external model wrapped and registered (migrate mode)

## 14. Open Questions & Potential Enhancements

### 14.1 — Decisions to Confirm Before Build

| # | Question | Current Assumption | Impact |
| --- | --- | --- | --- |
| 1 | Should `model_algorithm: "all"` run all 3 models and pick the best overall winner? | Yes — separate Optuna study per model, compare best of each | Longer runtime (~3x) but guaranteed best model |
| 2 | Should the framework auto-detect categorical vs numeric columns, or require user to specify? | Auto-detect from dtypes (object/string = categorical, numeric = numeric) | Edge case: numeric IDs misclassified as features |
| 3 | Should batch inference support incremental scoring (only new entities) or always full-table? | Full-table by default, incremental as config option | Cost vs freshness trade-off |
| 4 | Should we include an online table publishing step for low-latency serving feature lookup? | Not in v1 — enhancement for later | Serving latency will be higher without it |
| 5 | Should the framework include a model retraining trigger (e.g., when drift exceeds threshold)? | Not in v1 — monitor raises alert, human decides | Full automation vs human-in-the-loop |
| 6 | Log top-3 trial artifacts or strictly best-only? | Best-only (saves storage) | User can retrain any trial from logged params |

### 14.2 — Future Enhancements (v2+)

| Enhancement | Description | Complexity |
| --- | --- | --- |
| **A/B testing** | Route traffic between Champion and Challenger endpoints, measure performance | Medium |
| **Auto-retraining pipeline** | Trigger full retrain when drift_metric > threshold (via Databricks SQL Alert + Job) | Medium |
| **Online tables** | Publish feature table as online table for <10ms feature lookup in serving | Low |
| **Feature store lineage dashboard** | Auto-generate AI/BI dashboard showing feature freshness + lineage | Medium |
| **Multi-model ensemble** | Train multiple algorithms, combine via stacking or voting | High |
| **Time-series / forecasting mode** | Add Prophet/ARIMA support as a third `task_type` | High |
| ~~**Deep learning mode**~~ | ~~PyTorch/TensorFlow with GPU serverless compute~~ | **INCLUDED in v1** (see Section 4A) |
| ~~**Model explainability notebook**~~ | ~~SHAP values + feature importance as a standalone post-training step~~ | **INCLUDED in v1** (see Section 4B) |
| **Data quality checks** | Great Expectations or Databricks expectations before training | Medium |
| **Notification integration** | Slack/Teams/email on job completion, drift alert, or model promotion | Low |

### 14.3 — Known Constraints

| Constraint | Workaround |
| --- | --- |
| Serverless compute = single-node for ML training | `n_jobs` parallelism is driver-level only. For massive datasets, switch to classic cluster |
| `fe.score_batch()` requires model logged with FE Client | Framework always uses `fe.log_model()` — enforced by design |
| Online tables require DLT pipeline for sync | Not included in v1. Manual publish or future enhancement |
| Lakehouse Monitoring requires flat columns | `09_process_inference.py` flattens serving payload before monitoring |
| DABs require Databricks CLI installed | INSTALL notebook checks for CLI availability, prompts install if missing |

---

## 15. Build Sequence (after plan approval)

1. **INSTALL.py** — the entry point (config + generator + deploy)
2. **helpers.py** — shared utilities (config loader, deployment log, feature lookups)
3. **nn_architectures.py** — predefined DL architectures (MLP, ResNet, custom template)
4. **01_eda.py** — exploratory data analysis
5. **02_feature_engineering.py** — user-editable template with rich samples
6. **03_train_tune.py** — baselines + parallel Optuna HPO (tree models)
7. **03b_train_deep_learning.py** — PyTorch/TF neural network training + HPO
8. **04_evaluate.py** — holdout scoring + audit
9. **04b_explainability.py** — SHAP + permutation importance + PDP + local explanations
10. **05_register.py** — UC registration + aliases (picks best across tree AND DL)
11. **01_validate_model.py** (migrate) — pickle validation
12. **03_wrap_and_register.py** (migrate) — pyfunc wrapper
13. **06_batch_inference.py** — recurring scoring
14. **07_serve.py** — endpoint creation
15. **08_monitor.py** — Lakehouse Monitoring
16. **09_process_inference.py** — payload flattening
17. **databricks.yml** — DAB config
18. **resources/*.yml** — job definitions
19. **DEPLOYMENT_LOG.md** — audit template
20. **End-to-end test** on sample dataset

---

> **⭐ NEXT STEP:** Review this plan. Flag anything to add/remove/change. Once approved, I’ll build it sequentially starting with INSTALL.py.

## 16. In-Notebook Documentation Standard

Every generated notebook MUST be self-documenting. A new user who has never seen the framework should be able to open ANY notebook and understand: what it does, what it expects, what to edit, and what happens next.

---

### 16.1 — Mandatory Notebook Structure (every notebook follows this)

```
┌─────────────────────────────────────────────────────────────────────────┐
│  CELL 0: Header Markdown (auto-generated, never edited)                 │
│  ─────────────────────────────────────────────────────────────────────  │
│  • Notebook title + purpose (1-2 sentences)                             │
│  • Prerequisites (what must run before this notebook)                   │
│  • Inputs (tables, artifacts, config values consumed)                   │
│  • Outputs (tables, artifacts, metrics produced)                        │
│  • Estimated runtime                                                    │
│  • Next step (which notebook to run after this one)                     │
└─────────────────────────────────────────────────────────────────────────┘
┌─────────────────────────────────────────────────────────────────────────┐
│  CELL 1: Setup & Config (standard boilerplate, rarely edited)           │
└─────────────────────────────────────────────────────────────────────────┘
┌─────────────────────────────────────────────────────────────────────────┐
│  CELL N: Section Markdown (before each logical block)                   │
│  ─────────────────────────────────────────────────────────────────────  │
│  • What this section does                                               │
│  • Why it matters                                                       │
│  • What to look for in the output                                       │
└─────────────────────────────────────────────────────────────────────────┘
┌─────────────────────────────────────────────────────────────────────────┐
│  CELL N+1: Code (with inline comments explaining WHY, not WHAT)         │
└─────────────────────────────────────────────────────────────────────────┘
┌─────────────────────────────────────────────────────────────────────────┐
│  FINAL CELL: Summary Markdown                                           │
│  ─────────────────────────────────────────────────────────────────────  │
│  • What was accomplished                                                │
│  • Key outputs and where to find them                                   │
│  • What to do next                                                      │
└─────────────────────────────────────────────────────────────────────────┘
```

---

### 16.2 — Header Cell Template (Cell 0 of every notebook)

```markdown
# 03 — Train & Tune Model

## Purpose
Train baseline models and perform hyperparameter optimization using Optuna with 
parallel trials. Logs all experiments to MLflow. Only uses train/validation data — 
the test set is NEVER touched in this notebook.

## Prerequisites
- ✅ `01_eda.py` has been run (data profiled, no leakage detected)
- ✅ `02_feature_engineering.py` has been run (feature table exists and is populated)
- ✅ Config loaded with valid `training_table`, `feature_table_name`, `model_algorithm`

## Inputs
| Input | Source | Description |
| --- | --- | --- |
| Feature table | `config.feature_table_name` | Computed features in Unity Catalog |
| Training table | `config.training_table` | Entity + label + timestamp |
| Config | Job params or widget | Algorithm, n_trials, parallel settings |

## Outputs
| Output | Destination | Description |
| --- | --- | --- |
| MLflow experiment | `/Users/{user}/{model_name}_experiment` | All trials logged |
| Best model artifact | `runs:/{run_id}/model` | Full sklearn Pipeline with signature |
| Test split (saved) | Passed to 04_evaluate via temp table | Locked holdout set |

## Estimated Runtime
~5-15 minutes (50 trials, 4 parallel, ~10K rows)

## Next Step
→ Run `04_evaluate.py` to score holdout test set and validate training health
```

---

### 16.3 — Section Markdown Template (before each code block)

```markdown
## Step 3: Define Preprocessing Pipeline

**What this does:** Creates a sklearn ColumnTransformer that applies:
- StandardScaler to numeric features (normalizes to mean=0, std=1)
- OneHotEncoder to categorical features (converts to binary columns)

**Why it matters:** The preprocessing pipeline is part of the model artifact. 
Whatever transformations you define here will automatically run during batch 
inference and real-time serving — you never need to replicate this logic elsewhere.

**What to look for:** After this cell runs, you'll see the pipeline structure printed. 
Verify the correct columns are assigned to numeric vs categorical transformers.
```

---

### 16.4 — User-Editable Section Markers

Whenever user customization is expected, use this visual pattern:

```python
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  🔧 USER ACTION REQUIRED — Edit this section                           ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║                                                                        ║
# ║  Define your feature computation logic below.                          ║
# ║  Use Spark SQL or PySpark to read source tables and compute features.  ║
# ║                                                                        ║
# ║  Requirements:                                                         ║
# ║  • Output must contain the entity_key column (customer_id)             ║
# ║  • Output must contain the timestamp_key column (event_timestamp)      ║
# ║  • All other columns become features for the model                     ║
# ║                                                                        ║
# ║  See samples below — uncomment and adapt for your use case.            ║
# ║                                                                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
```

And after the user-editable section:

```python
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ✅ END OF USER-EDITABLE SECTION                                       ║
# ║  Everything below is managed by the framework — do not modify          ║
# ╚══════════════════════════════════════════════════════════════════════════╝
```

---

### 16.5 — Inline Comment Style Guide

**DO: Explain WHY (business logic, design decisions)**
```python
# Use PR-AUC instead of F1 because class imbalance > 5:1
# makes F1 unreliable (dominated by majority class)
objective_metric = "val_pr_auc" if imbalance_ratio > 5 else "val_f1"
```

**DO: Flag important constraints**
```python
# ⚠️ IMPORTANT: This test split must NEVER be used for tuning.
# It exists solely for final generalization assessment in 04_evaluate.py
test_df = full_df.filter(col("split") == "test")
```

**DO: Explain non-obvious behavior**
```python
# Limit model's internal parallelism to avoid CPU thrashing
# when running multiple Optuna trials simultaneously
cores_per_trial = max(1, os.cpu_count() // n_parallel_trials)
```

**DON'T: State the obvious**
```python
# BAD: "Load the config" — the code already says that
config = load_config()  # This is self-evident
```

---

### 16.6 — Validation Cells (with user-friendly output)

Every notebook includes validation cells that print clear status:

```python
# Validation output format:
print("═" * 60)
print("  ✅ VALIDATION PASSED")
print("═" * 60)
print(f"  Feature table: {feature_table_name}")
print(f"  Row count:     {row_count:,}")
print(f"  Columns:       {len(columns)}")
print(f"  Null in key:   0 (clean)")
print(f"  Entity key:    {entity_key} (unique: {n_unique:,})")
print("═" * 60)
```

Or on failure:
```python
print("═" * 60)
print("  ❌ VALIDATION FAILED")
print("═" * 60)
print(f"  Issue: Feature table '{feature_table_name}' is empty (0 rows)")
print(f"  Action: Run 02_feature_engineering.py first, then re-run this notebook")
print("═" * 60)
raise ValueError(f"Feature table is empty. Cannot proceed with training.")
```

---

### 16.7 — Complete Example: What 01_eda.py Looks Like to a New User

```
┌─────────────────────────────────────────────────────────────────────────┐
│  [MARKDOWN] # 01 — Exploratory Data Analysis                           │
│  Purpose, Prerequisites, Inputs, Outputs, Runtime, Next Step            │
├─────────────────────────────────────────────────────────────────────────┤
│  [CODE] # Setup & Config                                                │
│  Load config, print what we're analyzing                                │
├─────────────────────────────────────────────────────────────────────────┤
│  [MARKDOWN] ## Step 1: Load & Profile Data                              │
│  What this does, why it matters, what to look for                       │
├─────────────────────────────────────────────────────────────────────────┤
│  [CODE] # Load training table, display shape/dtypes/nulls               │
│  ... code with inline comments ...                                      │
│  Prints: "Training table: 64,231 rows × 23 columns"                    │
├─────────────────────────────────────────────────────────────────────────┤
│  [MARKDOWN] ## Step 2: Target Variable Distribution                     │
│  What this does, why (detect imbalance), what to look for               │
├─────────────────────────────────────────────────────────────────────────┤
│  [CODE] # Class balance visualization                                   │
│  ... code ...                                                           │
│  Prints: "⚠️ Class imbalance detected: 8.3:1 ratio"                    │
│  OR: "✅ Classes are balanced (1.2:1 ratio)"                            │
├─────────────────────────────────────────────────────────────────────────┤
│  [MARKDOWN] ## Step 3: Feature Correlations                             │
│  What this does, why (identify predictive signals), what to look for    │
├─────────────────────────────────────────────────────────────────────────┤
│  [CODE] # Correlogram + top-10 correlations with target                 │
│  ... code ...                                                           │
├─────────────────────────────────────────────────────────────────────────┤
│  [MARKDOWN] ## Step 4: Target Leakage Screen                            │
│  What this does (critical!), why, what triggers a STOP                  │
├─────────────────────────────────────────────────────────────────────────┤
│  [CODE] # Screen for leakage (|corr| > 0.95)                           │
│  ... code ...                                                           │
│  Prints: "✅ No target leakage detected"                                │
│  OR: "🛑 LEAKAGE DETECTED: 'revenue_this_month' (corr=0.98)"           │
├─────────────────────────────────────────────────────────────────────────┤
│  [MARKDOWN] ## Step 5: Available Source Tables                          │
│  Why (helps user pick feature sources for next notebook)                │
├─────────────────────────────────────────────────────────────────────────┤
│  [CODE] # List tables in schema with row counts                         │
│  ... code ...                                                           │
├─────────────────────────────────────────────────────────────────────────┤
│  [MARKDOWN] ## Summary                                                  │
│  "EDA complete. 64K rows, 23 features, no leakage.                      │
│   Class imbalance detected → framework will use PR-AUC during tuning.   │
│   → Next: open 02_feature_engineering.py to define your features"       │
└─────────────────────────────────────────────────────────────────────────┘
```

---

### 16.8 — Documentation Rules Summary

| Rule | Applies To | Purpose |
| --- | --- | --- |
| Header cell with Prerequisites/Inputs/Outputs/Next Step | Every notebook | New user knows context without reading code |
| Section markdown before each code block | Every notebook | Explains WHY before showing HOW |
| `🔧 USER ACTION REQUIRED` markers | 02_feature_engineering, 03_wrap_and_register | User knows exactly what to edit |
| `✅ END OF USER-EDITABLE SECTION` markers | Same | User knows what NOT to touch |
| Validation cells with ✅/❌ output | Every notebook | Clear pass/fail + actionable error messages |
| Inline comments (WHY, not WHAT) | All code cells | Design decisions are explained |
| Summary cell at end | Every notebook | Confirms what happened + next step |
| Estimated runtime in header | Every notebook | User knows what to expect |
| Emojis for visual scanning | Validation output only | Quick pass/fail recognition (not in code) |